# Семинар 2. Сегментация клиентов: RFM-анализ и K-Means кластеризация

**Цель семинара:** Научиться превращать «сырой» лог транзакций в бизнес-сегменты клиентов с помощью RFM-анализа и алгоритма K-Means. Результат этого семинара — колонка `Cluster_ID` в таблице клиентов — станет одним из ключевых признаков для модели оттока в Семинаре 5.

---

## Зачем вообще сегментировать клиентов?

Представьте, что вы управляете сетью магазинов. У вас 1000 клиентов. Вы хотите отправить им персональное предложение. Но у вас нет ресурсов, чтобы придумать 1000 уникальных сообщений.

**Решение:** разбить 1000 клиентов на 4 группы по поведению и сделать 4 разных предложения. Клиентам, которые давно не покупали — скидку. Лояльным «чемпионам» — программу лояльности. «Спящим» — напоминание о себе.

Именно это и делает RFM + K-Means. Никакой магии — только три цифры на клиента и математика расстояний.

---

## Что такое RFM?

RFM — это три метрики, каждая из которых описывает поведение клиента с разного угла:

| Буква | Название | Вопрос | Пример |
|-------|----------|--------|--------|
| **R** | Recency (Давность) | Сколько дней прошло с **последней** покупки? | 5 дней = активный; 200 дней = уходит |
| **F** | Frequency (Частота) | Сколько **раз** совершались операции? | 120 раз = лояльный; 20 раз = редкий |
| **M** | Monetary (Деньги) | Какова **суммарная** ценность клиента? | 6000 руб. = VIP; 800 руб. = случайный |

Идея простая: **лучший клиент** покупал недавно (R↓), часто (F↑) и на большие суммы (M↑).

---

## Данные семинара

Мы работаем с таблицей **transactions.csv** — логом операций 1000 клиентов за ~2 года. Это таблица фактов: на одного клиента приходится много строк, по одной на каждую транзакцию.

| Колонка | Тип | Описание |
|---------|-----|----------|
| `Target_ID` | int | Уникальный идентификатор клиента |
| `Trans_Date` | string → datetime | Дата и время транзакции |
| `Trans_Amount` | float | Сумма транзакции в рублях |

Наша задача — свернуть эти ~83 000 строк транзакций в таблицу из **1000 строк** (по одной на клиента) с тремя RFM-столбцами, а потом найти кластеры.

## 🧰 Шаг 0. Импорт библиотек

Сегодня нам понадобится расширенный набор инструментов:

- **pandas / numpy** — уже знакомы с Семинара 1
- **matplotlib / seaborn** — для визуализации
- **sklearn.preprocessing.StandardScaler** — нормализация данных перед кластеризацией
- **sklearn.cluster.KMeans** — сам алгоритм кластеризации
- **sklearn.decomposition.PCA** — для 2D-визуализации многомерных кластеров

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Все библиотеки успешно загружены!")
print("📦 Версии: pandas", pd.__version__, "| sklearn готов к работе")

## 📥 Шаг 1. Загрузка данных

Файл `transactions.csv` уже лежит в папке курса. Он содержит **83 469 строк** — каждая строка это одна транзакция одного клиента. Сначала просто прочитаем и осмотримся.

In [ ]:
# Путь к файлу (поправьте, если файл лежит в другом месте)
DATA_PATH = "data/seminar_2/transactions.csv"

# Загрузка данных
df = pd.read_csv(DATA_PATH)

print(f"📊 Загружено строк: {df.shape[0]:,}")
print(f"📋 Колонки: {df.columns.tolist()}")
print(f"👥 Уникальных клиентов: {df['Target_ID'].nunique()}")
print("\n🔍 Первые 5 строк:")
df.head()

> **Обратите внимание:** один и тот же `Target_ID` встречается много раз. Это и есть структура «таблицы фактов» — она хранит события, а не профили. Наша цель — сжать всё это в профиль каждого клиента.

---

## 🛠 ЗАДАНИЕ 1: Рентген транзакционных данных

Прежде чем считать метрики, нужно понять, с чем мы работаем. Посмотрим на типы данных и базовую статистику.

📋 **Что нужно сделать:**

- Вызовите `df.info()`, чтобы увидеть типы данных и количество непустых значений
- Вызовите `df.describe()`, чтобы получить статистику по числовым колонкам
- Проверьте, есть ли пропуски: `df.isnull().sum()`

🤖 **Застрял?** Отправь тьютору тег `#SEM2_TASK1_START` (если не знаешь какие методы вызывать) или `#SEM2_TASK1_WHY` (если методы сработали, но непонятно что означают числа).

In [ ]:
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

print("--- ПАСПОРТ ТАБЛИЦЫ ---")
# Вызовите метод info():


print("\n--- БАЗОВАЯ СТАТИСТИКА ---")
# Вызовите метод describe():


print("\n--- ПРОПУСКИ ---")
# Проверьте наличие пропусков:

---

## 🛠 ЗАДАНИЕ 2: Починим дату и найдем «точку отсчёта»

Вы должны были заметить, что `Trans_Date` имеет тип `object` (текст), а не `datetime`. Это проблема — мы не сможем вычислить, сколько дней прошло с последней транзакции, работая со строками.

Также нам нужна **точка отсчёта** для расчёта Recency — дата «сегодня» относительно датасета. Обычно берут максимальную дату в данных.

📋 **Что нужно сделать:**

- Преобразуйте колонку `Trans_Date` в формат datetime с помощью `pd.to_datetime()`
- Найдите максимальную дату в колонке и запишите её в переменную `snapshot_date`
- Выведите `snapshot_date` на экран

🤖 **Застрял?** Твои теги: `#SEM2_TASK2_START` (как конвертировать дату) или `#SEM2_TASK2_BUG` (если конвертация не работает).

In [ ]:
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

# Преобразуйте колонку Trans_Date в datetime:
df['Trans_Date'] = ...

# Найдите максимальную дату и запишите в snapshot_date:
snapshot_date = ...

print(f"📅 Точка отсчёта (snapshot): {snapshot_date}")
print(f"📅 Тип данных Trans_Date: {df['Trans_Date'].dtype}")

# Проверка
assert str(df['Trans_Date'].dtype).startswith('datetime'), "Ошибка: Trans_Date всё ещё не datetime!"
print("✅ Дата успешно преобразована!")

---

## 🛠 ЗАДАНИЕ 3: Вычисляем RFM-метрики

Это **центральное задание семинара**. Вам нужно «свернуть» 83 469 строк транзакций в 1000 строк — по одной на клиента — вычислив три метрики.

**Как считать каждую метрику:**

- **Recency** = `snapshot_date` минус дата **последней** транзакции клиента → результат в днях
- **Frequency** = **количество** транзакций клиента
- **Monetary** = **сумма** всех `Trans_Amount` клиента

📋 **Что нужно сделать:**

- Используйте `df.groupby('Target_ID').agg(...)` для агрегации
- Для Recency используйте lambda-функцию внутри agg
- Сбросьте индекс через `.reset_index()`
- Запишите результат в переменную `rfm`
- Выведите первые 10 строк

🤖 **Застрял?** Теги: `#SEM2_TASK3_START` (объясни как устроен groupby + agg) или `#SEM2_TASK3_BUG` (ошибка в lambda или типах данных).

In [ ]:
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

# Агрегируем: из ~83 000 строк → 1000 строк (по одной на клиента)
rfm = df.groupby('Target_ID').agg(
    Recency  = ('Trans_Date',   ...),   # дней с последней транзакции
    Frequency= ('Trans_Date',   ...),   # количество транзакций  
    Monetary = ('Trans_Amount', ...)    # суммарная ценность
).reset_index()

print(f"✅ RFM-таблица готова! Размер: {rfm.shape}")
print("\n📊 Первые 10 клиентов:")
rfm.head(10)

In [ ]:
# Посмотрим на статистику RFM
print("📈 Статистика RFM-метрик:")
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(1)

---

## 🛠 ЗАДАНИЕ 4: Нормализация — выравниваем игровое поле

Посмотрите на разброс значений: `Recency` измеряется в **днях** (0–270), `Frequency` — в **штуках** (16–141), а `Monetary` — в **рублях** (600–7000).

Если мы подадим эти сырые числа в K-Means, алгоритм будет думать, что разница в 100 рублей важнее, чем разница в 30 дней — просто потому что 100 > 30. Но это не так!

**Решение:** нормализация. `StandardScaler` вычитает среднее и делит на стандартное отклонение — после этого все три метрики живут в одном масштабе.

📋 **Что нужно сделать:**

- Создайте объект `scaler = StandardScaler()`
- Примените `scaler.fit_transform()` к колонкам `['Recency', 'Frequency', 'Monetary']` из таблицы `rfm`
- Запишите результат в `rfm_scaled`
- Выведите первые 5 строк, чтобы убедиться, что числа теперь около нуля

🤖 **Застрял?** Тег `#SEM2_TASK4_START` или `#SEM2_TASK4_WHY` (почему нормализация так важна).

In [ ]:
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

# Создайте объект StandardScaler:
scaler = ...

# Нормализуйте три RFM-колонки:
rfm_scaled = scaler.fit_transform(...)

print("✅ Нормализация выполнена!")
print(f"Форма: {rfm_scaled.shape}")
print("\nПервые 5 строк (значения должны быть около 0):")
print(rfm_scaled[:5].round(3))

# Проверка
assert rfm_scaled.shape == (1000, 3), "Ошибка: неправильная форма массива после нормализации!"
assert abs(rfm_scaled.mean()) < 0.01, "Ошибка: среднее должно быть близко к нулю после нормализации!"

---

## 🛠 ЗАДАНИЕ 5: Метод «локтя» — сколько кластеров нужно?

K-Means требует заранее указать количество кластеров `k`. Но как его выбрать? Нельзя просто взять 4 или 10 — нужно обосновать.

**Метод локтя (Elbow Method):**

Мы запускаем K-Means при разных значениях `k` (например, от 2 до 10) и для каждого записываем **инерцию** — сумму квадратов расстояний от каждой точки до центра её кластера. Чем меньше инерция — тем плотнее кластеры. Строим график и ищем «локоть» — точку, где кривая начинает выравниваться. Это и есть оптимальное `k`.

📋 **Что нужно сделать:**

- Создайте пустой список `inertias = []`
- В цикле `for k in range(2, 11)` обучите `KMeans(n_clusters=k, random_state=42, n_init=10)` на `rfm_scaled`
- Добавляйте `km.inertia_` в список
- Постройте график `plt.plot(range(2, 11), inertias, marker='o')` и визуально найдите «локоть»

🤖 **Застрял?** Теги: `#SEM2_TASK5_START` или `#SEM2_TASK5_WHY` (не понимаю, что такое инерция и как читать график).

In [ ]:
raise NotImplementedError("Задание 5 не выполнено! Удалите эту строку и напишите свой код.")

inertias = []

# Перебираем k от 2 до 10 включительно:
for k in range(2, 11):
    km = KMeans(n_clusters=..., random_state=42, n_init=10)
    km.fit(...)
    inertias.append(...)

# --- Код отрисовки (не меняйте) ---
plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), inertias, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.title('Метод локтя: выбор оптимального числа кластеров', fontsize=14, fontweight='bold')
plt.xlabel('Количество кластеров (k)', fontsize=12)
plt.ylabel('Инерция (Inertia)', fontsize=12)
plt.xticks(range(2, 11))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
# -----------------------------------

**❓ Вопрос для размышления:** Посмотрите на график. При каком значении `k` кривая начинает «ломаться» и выравниваться? Это и есть ваш «локоть».

*Запишите ваш ответ в текстовой ячейке ниже.*

**Мой ответ:** *(напишите здесь — при каком k вы видите локоть и почему)*

---

## 🛠 ЗАДАНИЕ 6: Обучаем финальную модель и присваиваем метки

По графику локтя оптимальное число кластеров для наших данных равно **4**. Обучим финальную модель и добавим метки сегментов обратно в таблицу клиентов.

📋 **Что нужно сделать:**

- Создайте `KMeans(n_clusters=4, random_state=42, n_init=10)` и обучите его на `rfm_scaled`
- Получите метки кластеров через `.labels_` (или `.predict()`)
- Добавьте их в таблицу `rfm` как новую колонку `Cluster_ID`
- Выведите количество клиентов в каждом кластере

🤖 **Застрял?** Тег `#SEM2_TASK6_START` или `#SEM2_TASK6_BUG`.

In [ ]:
raise NotImplementedError("Задание 6 не выполнено! Удалите эту строку и напишите свой код.")

# Обучаем финальную модель с k=4:
kmeans_final = KMeans(n_clusters=..., random_state=42, n_init=10)
kmeans_final.fit(...)

# Добавляем метки кластеров в таблицу rfm:
rfm['Cluster_ID'] = ...

print("✅ Кластеризация выполнена!")
print("\n📊 Клиентов в каждом кластере:")
print(rfm['Cluster_ID'].value_counts().sort_index())

# Проверка
assert 'Cluster_ID' in rfm.columns, "Ошибка: колонка Cluster_ID не найдена!"
assert rfm['Cluster_ID'].nunique() == 4, "Ошибка: должно быть ровно 4 кластера!"
print("\n✅ Проверка пройдена!")

---

## 📊 Шаг 7. Анализируем кластеры — кто есть кто?

Кластеры пронумерованы от 0 до 3, но эти числа ничего не значат сами по себе. Нужно понять **профиль** каждого кластера через средние значения RFM-метрик.

In [ ]:
# Средние RFM-метрики по каждому кластеру (этот код уже написан за вас)
cluster_profile = rfm.groupby('Cluster_ID')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
cluster_profile['Размер'] = rfm['Cluster_ID'].value_counts().sort_index()

print("📊 Профиль кластеров:")
print(cluster_profile)

# Визуализация профилей
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['Recency', 'Frequency', 'Monetary']
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for i, metric in enumerate(metrics):
    vals = cluster_profile[metric]
    axes[i].bar(vals.index.astype(str), vals.values, color=colors)
    axes[i].set_title(f'{metric} по кластерам', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Кластер', fontsize=11)
    axes[i].set_ylabel(metric, fontsize=11)
    for j, v in enumerate(vals.values):
        axes[i].text(j, v * 1.01, f'{v:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Портрет каждого поведенческого сегмента', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 🛠 ЗАДАНИЕ 7: Дайте кластерам бизнес-имена

Посмотрите на профиль каждого кластера в таблице выше. У каждого есть характер:

- Маленький Recency + высокая Frequency + высокая Monetary = **лояльный активный клиент**
- Большой Recency + низкая Frequency + низкая Monetary = **уходящий или потерянный клиент**

📋 **Что нужно сделать:**

- Изучите таблицу `cluster_profile` выше
- Заполните словарь `segment_names` ниже, дав каждому кластеру (0–3) осмысленное бизнес-название
- Добавьте колонку `Segment` в таблицу `rfm`

**Примеры названий:** «Чемпионы», «Лояльные», «Группа риска», «Спящие», «Новички», «Потерянные», «VIP».

🤖 **Застрял?** Отправь тьютору `#SEM2_TASK7_WHY` — как понять, какое название подходит какому кластеру.

In [ ]:
raise NotImplementedError("Задание 7 не выполнено! Удалите эту строку и напишите свой код.")

# Заполните названия кластеров на основе анализа профиля выше:
segment_names = {
    0: "...",   # Смотри на R, F, M кластера 0 и дай название
    1: "...",   # Смотри на R, F, M кластера 1
    2: "...",   # Смотри на R, F, M кластера 2
    3: "...",   # Смотри на R, F, M кластера 3
}

# Добавляем колонку с именем сегмента:
rfm['Segment'] = rfm['Cluster_ID'].map(segment_names)

print("✅ Сегменты присвоены!")
print("\n📋 Итоговая таблица (первые 10 строк):")
rfm.head(10)

---

## 📊 Шаг 8. Визуализация кластеров в 2D через PCA

RFM — это трёхмерное пространство. Чтобы нарисовать кластеры на плоскости, мы применяем **PCA (Principal Component Analysis)** — метод, который сжимает 3 измерения в 2, сохраняя максимум информации. Это готовый код для иллюстрации — просто запустите его.

In [ ]:
# PCA: 3D → 2D для визуализации (код написан за вас)
pca = PCA(n_components=2, random_state=42)
rfm_2d = pca.fit_transform(rfm_scaled)

rfm_plot = rfm.copy()
rfm_plot['PC1'] = rfm_2d[:, 0]
rfm_plot['PC2'] = rfm_2d[:, 1]

plt.figure(figsize=(11, 7))
palette = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for cid in sorted(rfm_plot['Cluster_ID'].unique()):
    mask = rfm_plot['Cluster_ID'] == cid
    label = f"Кластер {cid}: {rfm_plot.loc[mask, 'Segment'].iloc[0]}"
    plt.scatter(
        rfm_plot.loc[mask, 'PC1'],
        rfm_plot.loc[mask, 'PC2'],
        label=label, alpha=0.7, s=60, color=palette[cid]
    )

plt.title('Сегменты клиентов (PCA-проекция RFM-пространства)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 (объясняет {pca.explained_variance_ratio_[0]*100:.1f}% дисперсии)', fontsize=11)
plt.ylabel(f'PC2 (объясняет {pca.explained_variance_ratio_[1]*100:.1f}% дисперсии)', fontsize=11)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

total_var = sum(pca.explained_variance_ratio_) * 100
print(f"\n📊 Два компонента объясняют {total_var:.1f}% дисперсии исходных данных")

---

## 🛠 ЗАДАНИЕ 8: Экспорт результата

Финальный аккорд. Колонка `Cluster_ID` из нашей таблицы `rfm` — это готовый признак для модели оттока. В Семинаре 5 мы объединим её с таблицей профилей клиентов.

📋 **Что нужно сделать:**

- Сохраните таблицу `rfm` в файл `data/seminar_2/rfm_with_clusters.csv` с помощью `df.to_csv()`, **без** индекса (`index=False`)
- Убедитесь, что файл создан, выведите его размер

🤖 **Застрял?** Тег `#SEM2_TASK8_START`.

In [ ]:
raise NotImplementedError("Задание 8 не выполнено! Удалите эту строку и напишите свой код.")

OUTPUT_PATH = "data/seminar_2/rfm_with_clusters.csv"

# Создайте папку если не существует (этот код написан за вас):
os.makedirs("data/seminar_2", exist_ok=True)

# Сохраните таблицу rfm в CSV:
rfm.to_csv(..., index=...)

# Проверка:
saved = pd.read_csv(OUTPUT_PATH)
print(f"✅ Файл сохранён: {OUTPUT_PATH}")
print(f"📊 Строк: {saved.shape[0]} | Колонок: {saved.shape[1]}")
print(f"📋 Колонки: {saved.columns.tolist()}")

assert 'Cluster_ID' in saved.columns, "Ошибка: в сохранённом файле нет колонки Cluster_ID!"
assert saved.shape[0] == 1000, "Ошибка: должно быть 1000 клиентов!"
print("\n✅ Файл готов для Семинара 5!")

---

## 🏁 Итоги семинара и критерии оценки

Поздравляю! Вы прошли полный цикл сегментации клиентов:

1. Загрузили лог из 83 469 транзакций
2. Агрегировали его в RFM-таблицу: 1000 клиентов × 3 метрики
3. Нормализовали данные для корректной работы алгоритма
4. Выбрали оптимальное число кластеров методом локтя
5. Обучили K-Means и получили 4 поведенческих сегмента
6. Дали кластерам бизнес-имена и визуализировали результат
7. Сохранили результат для следующих семинаров

---

### 📋 Что должно быть в вашем финальном ноутбуке:

- Данные загружены, дата преобразована в datetime
- Вычислены все три RFM-метрики через `groupby().agg()`
- Данные нормализованы через `StandardScaler`
- Построен и осмыслен график метода локтя
- K-Means обучен с `k=4`, колонка `Cluster_ID` добавлена в таблицу
- Каждому кластеру дано осмысленное бизнес-название
- Файл `rfm_with_clusters.csv` сохранён
- Написан текстовый вывод в ячейке ниже

---

### 🎯 Критерии успешной сдачи:

- **🟢 Техническая работоспособность:** ноутбук выполняется без ошибок от начала до конца, `NotImplementedError` нигде не остался
- **🧮 Корректность RFM:** в таблице `rfm` ровно 1000 строк и корректные метрики (Recency ≥ 0, Frequency > 0, Monetary > 0)
- **💼 Бизнес-интерпретация:** написан текстовый вывод (см. ячейку ниже) — своими словами, 3–5 предложений

### ✍️ Мой бизнес-вывод

*Ответьте на следующие вопросы своими словами (3–5 предложений):*

*Какие 4 сегмента получились в итоге и чем они отличаются? Какому сегменту компания должна уделить приоритетное внимание с точки зрения удержания клиентов? Какое маркетинговое действие вы бы предложили для каждого сегмента?*

**Мой ответ:** *(напишите здесь)*

---

## 🚀 Автопроверка

In [ ]:
def run_autocheck():
    print("🚀 Запуск автоматической проверки...\n" + "-" * 45)
    passed = True

    # 1. Проверка загрузки данных
    try:
        if not isinstance(df, pd.DataFrame):
            print("❌ Ошибка: переменная 'df' не является DataFrame.")
            passed = False
        elif df.shape[0] < 80000:
            print("❌ Ошибка: в таблице транзакций слишком мало строк. Вы загрузили правильный файл?")
            passed = False
        else:
            print(f"✅ Данные загружены корректно ({df.shape[0]:,} транзакций).")
    except NameError:
        print("❌ Ошибка: переменная 'df' не найдена. Выполните Задание 1.")
        passed = False

    # 2. Проверка типа даты
    try:
        if not str(df['Trans_Date'].dtype).startswith('datetime'):
            print("❌ Ошибка: колонка Trans_Date не имеет тип datetime. Выполните Задание 2.")
            passed = False
        else:
            print("✅ Колонка Trans_Date корректно преобразована в datetime.")
    except Exception as e:
        print(f"❌ Ошибка при проверке даты: {e}")
        passed = False

    # 3. Проверка RFM-таблицы
    try:
        if rfm.shape[0] != 1000:
            print(f"❌ Ошибка: в RFM-таблице {rfm.shape[0]} строк, должно быть 1000.")
            passed = False
        elif not all(col in rfm.columns for col in ['Recency', 'Frequency', 'Monetary']):
            print("❌ Ошибка: в таблице rfm не хватает колонок Recency/Frequency/Monetary.")
            passed = False
        elif rfm['Recency'].min() < 0:
            print("❌ Ошибка: Recency не может быть отрицательным. Проверьте формулу.")
            passed = False
        else:
            print(f"✅ RFM-таблица корректна: {rfm.shape[0]} клиентов, все метрики на месте.")
    except NameError:
        print("❌ Ошибка: переменная 'rfm' не найдена. Выполните Задание 3.")
        passed = False

    # 4. Проверка нормализации
    try:
        if rfm_scaled.shape != (1000, 3):
            print(f"❌ Ошибка: неверная форма rfm_scaled: {rfm_scaled.shape}. Ожидается (1000, 3).")
            passed = False
        elif abs(rfm_scaled.mean()) > 0.05:
            print("❌ Ошибка: среднее нормализованных данных слишком далеко от нуля.")
            passed = False
        else:
            print("✅ Нормализация выполнена корректно.")
    except NameError:
        print("❌ Ошибка: переменная 'rfm_scaled' не найдена. Выполните Задание 4.")
        passed = False

    # 5. Проверка кластеризации
    try:
        if 'Cluster_ID' not in rfm.columns:
            print("❌ Ошибка: колонка Cluster_ID не найдена в таблице rfm.")
            passed = False
        elif rfm['Cluster_ID'].nunique() != 4:
            print(f"❌ Ошибка: найдено {rfm['Cluster_ID'].nunique()} кластеров, ожидается 4.")
            passed = False
        else:
            print(f"✅ Кластеризация выполнена: 4 кластера присвоены всем {rfm.shape[0]} клиентам.")
    except Exception as e:
        print(f"❌ Ошибка при проверке кластеризации: {e}")
        passed = False

    # 6. Проверка сегментов
    try:
        if 'Segment' not in rfm.columns:
            print("❌ Ошибка: колонка Segment не найдена. Выполните Задание 7.")
            passed = False
        elif rfm['Segment'].isnull().any():
            print("❌ Ошибка: в колонке Segment есть пустые значения. Проверьте словарь segment_names.")
            passed = False
        elif any(v.strip() == '...' for v in rfm['Segment'].unique()):
            print("❌ Ошибка: замените '...' на реальные названия сегментов в словаре segment_names.")
            passed = False
        else:
            print(f"✅ Сегменты присвоены: {rfm['Segment'].unique().tolist()}")
    except Exception as e:
        print(f"❌ Ошибка при проверке сегментов: {e}")
        passed = False

    # 7. Проверка экспорта
    try:
        if not os.path.exists("data/seminar_2/rfm_with_clusters.csv"):
            print("❌ Ошибка: файл rfm_with_clusters.csv не найден. Выполните Задание 8.")
            passed = False
        else:
            saved_check = pd.read_csv("data/seminar_2/rfm_with_clusters.csv")
            if saved_check.shape[0] != 1000:
                print(f"❌ Ошибка: в сохранённом файле {saved_check.shape[0]} строк, должно быть 1000.")
                passed = False
            else:
                print("✅ Файл rfm_with_clusters.csv успешно сохранён и прочитан.")
    except Exception as e:
        print(f"❌ Ошибка при проверке экспорта: {e}")
        passed = False

    print("-" * 45)
    if passed:
        print("🎉 ПОЗДРАВЛЯЕМ! Техническая часть семинара выполнена безупречно!")
        print("Остался последний шаг: напишите ваш бизнес-вывод в текстовой ячейке выше.")
    else:
        print("⚠️ Обнаружены недочёты. Исправьте их и запустите проверку ещё раз!")

# Запуск проверки
run_autocheck()